In [19]:
import re
import string
import emoji
import contractions
from textblob import TextBlob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI


In [54]:
data = open('RAG_Text_Normalization_Document.txt', encoding='utf-8').read()
# print(open('Text_Normalization_and_RAG_Pipeline_Notes.txt', encoding='utf-8').read()) # text in the file

In [55]:
print(data[:100])
data = data.lower()
print(data[:100])

Artificial Intelligence (AI)     is a brnch of computer science that
focuses on building systems cap
artificial intelligence (ai)     is a brnch of computer science that
focuses on building systems cap


In [56]:
print(data[:100])
print()
data = re.sub(r'\s{2,}',' ', data)  # remove extra spaces
print(data[:100])

artificial intelligence (ai)     is a brnch of computer science that
focuses on building systems cap

artificial intelligence (ai) is a brnch of computer science that
focuses on building systems capable


In [57]:
print(data[:100])
print()
data = contractions.fix(data)  # expand contractions and abbreviations
print(data[:100])

artificial intelligence (ai) is a brnch of computer science that
focuses on building systems capable

artificial intelligence (ai) is a brnch of computer science that
focuses on building systems capable


In [58]:
print(data[:100])
print()
data = re.sub(r'[^0-9a-zA-Z\s]','', data) # remove punctuations and special characters
print(data[:100])

artificial intelligence (ai) is a brnch of computer science that
focuses on building systems capable

artificial intelligence ai is a brnch of computer science that
focuses on building systems capable o


In [59]:
print(data[:100])
print()
data = str(TextBlob(data).correct()) # correct spellings
print(data[:100])

artificial intelligence ai is a brnch of computer science that
focuses on building systems capable o



artificial intelligence ai is a branch of computer science that
focused on building systems capable 


In [60]:
import spacy
nlp = spacy.load("en_core_web_sm") # <spacy.lang.en.English at 0x20b48a2f0e0>

In [61]:
tokens = nlp(data) # returns the full text

In [62]:
# for token in tokens:
    # print(token,' ',token.lemma_)
    # print(token, ' ', token.is_stop)

# artificial   artificial
# intelligence   intelligence
# ai   ai
# is   be
# a   a
# branch   branch


# artificial   False
# intelligence   False
# ai   False
# is   True
# a   True
# branch   False
# of   True
# computer   False
# science   False
# that   True

In [63]:
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]

In [64]:
# updated_tokens[:100]    
# ['artificial',
#  'intelligence',
#  'ai',
#  'branch',
#  'computer',
#  'science',
#  '\n',
#  'focus',

In [65]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 20)
# <langchain_text_splitters.character.RecursiveCharacterTextSplitter at 0x20b4a148ec0>

In [66]:
chunks = splitter.create_documents([data])
# [Document(metadata={}, page_content='artificial intelligence ai is a branch of computer science that'),
#  Document(metadata={}, page_content='focused on building systems capable of performing tasks that normally'),


In [71]:
embedding_model = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L6-V2')
# Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2889.04it/s]
# HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-V2', cache_folder=None, 
# model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4745.57it/s]


In [73]:
vectordb = FAISS.from_documents(documents = chunks, embedding = embedding_model) 
# <langchain_community.vectorstores.faiss.FAISS at 0x20b56c54ad0>

In [74]:
query = "What is Artificial Interlligence?"

In [77]:
R_chunks = vectordb.similarity_search(query, k = 2)

# [Document(id='894e9f75-9ee6-474c-b419-7dba09734af2', metadata={}, page_content='artificial intelligence ai is a branch of computer science that'),
#  Document(id='d99da237-e9a1-4141-8471-6aaddfd2c930', metadata={}, page_content='making machine learning my is a sunset of artificial intelligence instead of')]

In [88]:
R_chunks = {chunk.page_content for chunk in R_chunks}
# {'artificial intelligence ai is a branch of computer science that',
#  'making machine learning my is a sunset of artificial intelligence instead of'}

In [ ]:
R_text = '\n'.join(R_chunks)
# 'artificial intelligence ai is a branch of computer science that\nmaking machine learning my is a sunset of artificial intelligence instead of'

'artificial intelligence ai is a branch of computer science that\nmaking machine learning my is a sunset of artificial intelligence instead of'

In [103]:
prompt = '''
Answer the following question from the retrieved context.
Structure the output in this format

Output Structure:

Question: "What is Artificial Interlligence?"
Answer: structured output

'''

In [92]:
import os
llm_model = ChatGoogleGenerativeAI(model="gemini-3.5-flash", api_key=os.environ["GEMINI_API_KEY"])

# ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.2.12', 
#                                                  'langchain-google-genai': '4.3.2'}}, 
#                                                  profile={'name': 'Gemini 3.5 Flash', 
#                                                           'release_date': '2026-05-19', 
#                                                           'last_updated': '2026-05-19', 
#                                                           'open_weights': False, 
#                                                           'max_input_tokens': 1048576, 
#                                                           'max_output_tokens': 65536, 
#                                                           'text_inputs': True, 
#                                                           'image_inputs': True, 
#                                                           'audio_inputs': True, 
#                                                           'pdf_inputs': True, 
#                                                           'video_inputs': True, 
#                                                           'text_outputs': True, 
#                                                           'image_outputs': False, 
#                                                           'audio_outputs': False, 
#                                                           'video_outputs': False, 
#                                                           'reasoning_output': True, 
#                                                           'tool_calling': True, 
#                                                           'structured_output': True, 
#                                                           'attachment': True, 
#                                                           'temperature': True, 
#                                                           'image_url_inputs': True, 
#                                                           'image_tool_message': True, 
#                                                           'tool_choice': True, 
#                                                           'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 
#                                                           'reasoning_effort_default': 'medium'}, 
#                                                           google_api_key=SecretStr('**********'), 
#                                                           model='gemini-3.5-flash', 
#                                                           temperature=None, 
#                                                           client=<google.genai.client.Client object at 0x0000020B56C55A90>, 
#                                                           default_metadata=(), model_kwargs={})

In [94]:
# llm_model.invoke(prompt)

# AIMessage(content=[{'type': 'text', 'text': 'Question: {query}\nAnswer: 
# Please provide the question and the retrieved context, and I will gladly answer 
# it for you in this format.', 'extras': {'signature': 
# 'EsUSCsISARFNMg9vv3FDoiytr9sMqE6ufa/JH41UP7TT9Mwt7g2p+HmN....

In [102]:
# llm_model.invoke(prompt).content
# [{'type': 'text',
#   'text': 'Question: {query}\nAnswer: Please provide the question and the retrieved context so that I can generate the correct structured answer for you.',
#   'extras': {'signature': 'EowQCokQAR

In [104]:
response = llm_model.invoke(prompt).content
print(response[0]['text'])

Question: "What is Artificial Interlligence?"
Answer: Artificial Intelligence (AI) is a field of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, problem-solving, decision-making, and understanding natural language.
